# Gradient-Based Optimization for Carbon Capture

This notebook demonstrates the key advantage of difflow: **automatic differentiation**
through chemical process models for gradient-based optimization.

## Learning Objectives

1. Understand why gradients matter for process optimization
2. Use JAX's `grad` to compute derivatives through unit operations
3. Optimize capture processes using gradient descent
4. Perform sensitivity analysis efficiently with `vmap`
5. Compare gradient-based vs derivative-free optimization

## 1. Why Gradients Matter

### Traditional Process Optimization

Chemical process optimization typically uses:
- **Derivative-free methods**: Nelder-Mead, genetic algorithms, particle swarm
- **Finite differences**: Approximate gradients by perturbing parameters

Problems:
- Derivative-free methods scale poorly with dimension
- Finite differences are inaccurate and expensive ($O(n)$ evaluations per gradient)

### Automatic Differentiation (AD)

AD computes **exact gradients** at cost similar to a single function evaluation!

$$\nabla f(x) = \left(\frac{\partial f}{\partial x_1}, \frac{\partial f}{\partial x_2}, \ldots, \frac{\partial f}{\partial x_n}\right)$$

This enables:
- Efficient high-dimensional optimization
- Sensitivity analysis
- Uncertainty quantification
- Integration with machine learning

## 2. Setup

In [ ]:
import jax
import jax.numpy as jnp
from jax import grad, jit, vmap, value_and_grad

jax.config.update("jax_enable_x64", True)

from difflow.streams import make_stream, get_flows, total_flow

from difflow_cc import (
    # Amine absorption
    AbsorberParams, AmineAbsorber,
    # Membranes
    MembraneParams, MembraneSeparator,
    # Adsorption
    AdsorptionParams, PSAUnit, VSAUnit,
)

print(f"JAX version: {jax.__version__}")
print(f"Using 64-bit floats: {jax.config.jax_enable_x64}")

## 3. Example 1: Optimizing L/G Ratio in Amine Absorption

### Problem Statement

Find the optimal liquid-to-gas (L/G) ratio that minimizes:
$$\text{Cost} = \text{Solvent cost} + \text{Penalty for missed capture}$$

- Higher L/G → better capture, but more solvent (regeneration energy)
- Lower L/G → less solvent, but may miss capture target

In [ ]:
# Define the flue gas
flue_gas = make_stream(
    flows={"CO2": 1.0, "N2": 6.67},  # 15% CO2 typical coal flue gas
    T=313.15,  # 40°C
    P=101325.0,
)

# Create absorber with variable L/G
def create_absorber(L_G_ratio):
    params = AbsorberParams(
        solvent='MEA',
        n_stages=10,
        solvent_conc=30.0,
        L_G_ratio=L_G_ratio,
        lean_loading=0.2,
    )
    return AmineAbsorber(params)

# Cost function to minimize
def capture_cost(L_G_ratio):
    """Total cost = solvent + capture penalty."""
    absorber = create_absorber(L_G_ratio)
    _, _, info = absorber(flue_gas)
    
    capture_eff = info['capture_efficiency']
    
    # Costs (simplified)
    solvent_cost = L_G_ratio * 10.0  # $/hr per L/G unit
    capture_target = 0.90
    penalty = 1000.0 * jnp.maximum(0, capture_target - capture_eff)**2
    
    return solvent_cost + penalty

# Test the function
L_G_test = 3.0
cost = capture_cost(L_G_test)
print(f"Cost at L/G = {L_G_test}: ${float(cost):.2f}/hr")

In [ ]:
# Compute gradient using JAX
grad_cost = grad(capture_cost)

# Gradient at test point
d_cost_d_LG = grad_cost(L_G_test)
print(f"Gradient at L/G = {L_G_test}: {float(d_cost_d_LG):.4f} $/hr per unit L/G")

if d_cost_d_LG > 0:
    print("  → Decreasing L/G will reduce cost")
else:
    print("  → Increasing L/G will reduce cost")

In [ ]:
# Simple gradient descent optimization
def gradient_descent(f, x0, learning_rate=0.1, n_iters=50):
    """Simple gradient descent with momentum."""
    x = x0
    grad_f = grad(f)
    history = []
    
    for i in range(n_iters):
        fx = f(x)
        gx = grad_f(x)
        history.append((float(x), float(fx), float(gx)))
        
        # Update with gradient
        x = x - learning_rate * gx
        
        # Keep L/G in reasonable range
        x = jnp.clip(x, 1.0, 10.0)
        
        if i % 10 == 0:
            print(f"Iter {i:3d}: L/G = {float(x):.3f}, Cost = ${float(fx):.2f}, Grad = {float(gx):.4f}")
    
    return x, history

# Optimize
L_G_init = 5.0
L_G_opt, history = gradient_descent(capture_cost, L_G_init, learning_rate=0.05, n_iters=50)

print(f"\nOptimal L/G ratio: {float(L_G_opt):.3f}")

In [ ]:
# Verify the optimum
absorber_opt = create_absorber(L_G_opt)
_, _, info_opt = absorber_opt(flue_gas)

print(f"At optimal L/G = {float(L_G_opt):.3f}:")
print(f"  Capture efficiency: {float(info_opt['capture_efficiency']):.1%}")
print(f"  Total cost: ${float(capture_cost(L_G_opt)):.2f}/hr")

## 4. Example 2: Multi-Parameter Membrane Optimization

Optimize both **membrane area** and **pressure ratio** to minimize:
$$\text{Total cost} = \text{Capital (area)} + \text{Operating (compression)} - \text{CO}_2\text{ revenue}$$

In [ ]:
# Feed gas
feed_gas = make_stream(
    flows={"CO2": 1.0, "N2": 4.0},  # 20% CO2
    T=298.15,
    P=500000.0,  # 5 bar feed
)

def membrane_cost(params):
    """Annualized cost of membrane separation.
    
    Args:
        params: [area_m2, P_permeate_Pa]
    """
    area = params[0]
    P_perm = params[1]
    
    mem_params = MembraneParams(
        membrane="PDMS_Rubbery",
        area=area,
        P_feed=500000.0,
        P_permeate=P_perm,
    )
    membrane = MembraneSeparator(mem_params)
    _, permeate, info = membrane(feed_gas)
    
    # Capital cost (area-based)
    membrane_cost_per_m2 = 100.0  # $/m²
    capital = area * membrane_cost_per_m2 / 5  # Annualized over 5 years
    
    # Operating cost (compression for vacuum)
    pressure_ratio = 500000.0 / (P_perm + 1.0)
    compression_cost = 0.05 * jnp.log(pressure_ratio) * 1000  # $/yr
    
    # Revenue from CO2 (negative cost)
    CO2_captured = info['CO2_recovery'] * 1.0  # mol/s
    CO2_revenue = CO2_captured * 44.0 * 3600 * 8000 * 50 / 1e6  # 50 $/tonne, 8000 hr/yr
    
    # Penalty for low purity
    purity = info['permeate_CO2_purity']
    purity_penalty = 10000.0 * jnp.maximum(0, 0.7 - purity)**2
    
    total = capital + compression_cost - CO2_revenue + purity_penalty
    return total

# Test
params_test = jnp.array([10.0, 50000.0])  # 10 m², 0.5 bar permeate
cost = membrane_cost(params_test)
print(f"Cost at area=10m², P_perm=0.5bar: ${float(cost):.0f}/yr")

In [ ]:
# Compute gradient w.r.t. both parameters
grad_membrane = grad(membrane_cost)

grads = grad_membrane(params_test)
print(f"Gradient at test point:")
print(f"  d(cost)/d(area)     = {float(grads[0]):.2f} $/yr per m²")
print(f"  d(cost)/d(P_perm)   = {float(grads[1]):.6f} $/yr per Pa")

In [ ]:
# Multi-parameter gradient descent
def optimize_membrane(n_iters=100):
    params = jnp.array([10.0, 50000.0])
    learning_rates = jnp.array([0.5, 5000.0])  # Different scales
    
    grad_f = grad(membrane_cost)
    
    for i in range(n_iters):
        cost = membrane_cost(params)
        grads = grad_f(params)
        
        # Gradient descent update
        params = params - learning_rates * grads
        
        # Enforce bounds
        params = jnp.array([
            jnp.clip(params[0], 1.0, 100.0),   # Area: 1-100 m²
            jnp.clip(params[1], 10000.0, 200000.0),  # P_perm: 0.1-2 bar
        ])
        
        if i % 20 == 0:
            print(f"Iter {i:3d}: Area={float(params[0]):.1f}m², "
                  f"P_perm={float(params[1])/1e5:.2f}bar, "
                  f"Cost=${float(cost):.0f}/yr")
    
    return params

params_opt = optimize_membrane()
print(f"\nOptimal: Area={float(params_opt[0]):.1f}m², P_perm={float(params_opt[1])/1e5:.2f}bar")

## 5. Sensitivity Analysis with `vmap`

Use JAX's `vmap` (vectorized map) to efficiently compute sensitivities
across many parameter values in parallel.

In [ ]:
# Vectorized sensitivity analysis for L/G ratio
@jit
def capture_efficiency(L_G):
    absorber = create_absorber(L_G)
    _, _, info = absorber(flue_gas)
    return info['capture_efficiency']

# Compute efficiency and gradient for many L/G values at once
L_G_values = jnp.linspace(1.0, 8.0, 50)

# Vectorize both function and gradient
capture_eff_vec = vmap(capture_efficiency)
grad_capture = grad(capture_efficiency)
grad_capture_vec = vmap(grad_capture)

# Compute all at once
efficiencies = capture_eff_vec(L_G_values)
sensitivities = grad_capture_vec(L_G_values)

print(f"{'L/G':>6} {'Capture':>10} {'Sensitivity':>12}")
print("-" * 30)
for i in range(0, 50, 10):
    print(f"{float(L_G_values[i]):>6.2f} {float(efficiencies[i]):>10.1%} "
          f"{float(sensitivities[i]):>12.4f}")

In [ ]:
# Find inflection point (where sensitivity is highest)
max_sens_idx = jnp.argmax(jnp.abs(sensitivities))
print(f"\nMaximum sensitivity at L/G = {float(L_G_values[max_sens_idx]):.2f}")
print(f"  Sensitivity = {float(sensitivities[max_sens_idx]):.4f}")
print(f"  This is where small changes in L/G have biggest effect on capture")

## 6. Comparing Optimization Approaches

Let's compare gradient-based optimization with a derivative-free method.

In [ ]:
import time

# Count function evaluations
eval_count = 0

def counted_cost(L_G):
    global eval_count
    eval_count += 1
    return capture_cost(L_G)

# Gradient descent
eval_count = 0
start = time.time()
L_G_gd, _ = gradient_descent(capture_cost, 5.0, learning_rate=0.05, n_iters=30)
gd_time = time.time() - start
# Each iteration: 1 forward + 1 backward ≈ 2 evals
gd_evals = 30 * 2

print(f"Gradient Descent:")
print(f"  Optimal L/G: {float(L_G_gd):.3f}")
print(f"  Iterations: 30")
print(f"  Equiv. evaluations: ~{gd_evals}")
print(f"  Time: {gd_time:.3f}s")

In [ ]:
# Golden section search (derivative-free)
def golden_section(f, a, b, tol=0.01):
    """Golden section search for unimodal function."""
    phi = (1 + 5**0.5) / 2
    
    c = b - (b - a) / phi
    d = a + (b - a) / phi
    
    evals = 0
    while abs(b - a) > tol:
        fc = f(c)
        fd = f(d)
        evals += 2
        
        if fc < fd:
            b = d
            d = c
            c = b - (b - a) / phi
        else:
            a = c
            c = d
            d = a + (b - a) / phi
    
    return (a + b) / 2, evals

start = time.time()
L_G_gs, gs_evals = golden_section(capture_cost, 1.0, 10.0, tol=0.01)
gs_time = time.time() - start

print(f"\nGolden Section Search:")
print(f"  Optimal L/G: {float(L_G_gs):.3f}")
print(f"  Evaluations: {gs_evals}")
print(f"  Time: {gs_time:.3f}s")

In [ ]:
# Summary comparison
print("\nComparison Summary:")
print(f"{'Method':<25} {'Optimal L/G':<12} {'Evals':<10}")
print("-" * 47)
print(f"{'Gradient Descent':<25} {float(L_G_gd):<12.3f} {'~60':10}")
print(f"{'Golden Section':<25} {float(L_G_gs):<12.3f} {gs_evals:<10}")
print()
print("Note: For 1D problems, derivative-free methods work well.")
print("For high-dimensional problems, gradients become essential!")

## 7. Advanced: Jacobian and Hessian

JAX can compute full Jacobians and Hessians for more sophisticated optimization.

In [ ]:
from jax import jacfwd, jacrev, hessian

# Function with vector output
def absorber_outputs(L_G):
    absorber = create_absorber(L_G)
    _, _, info = absorber(flue_gas)
    return jnp.array([
        info['capture_efficiency'],
        info['rich_loading'],
    ])

# Jacobian: derivatives of all outputs w.r.t. input
jac = jacfwd(absorber_outputs)
J = jac(3.0)

print("Jacobian at L/G = 3.0:")
print(f"  d(capture_eff)/d(L/G)  = {float(J[0]):.4f}")
print(f"  d(rich_loading)/d(L/G) = {float(J[1]):.4f}")

In [ ]:
# Hessian for second-order optimization (Newton's method)
hess_cost = hessian(capture_cost)
H = hess_cost(3.0)

print(f"Hessian (curvature) at L/G = 3.0: {float(H):.4f}")

if H > 0:
    print("  Positive curvature → local minimum region")
else:
    print("  Negative curvature → local maximum region")

## 8. Key Takeaways

1. **difflow enables gradient-based optimization** of carbon capture processes

2. **JAX's `grad` function** computes exact derivatives through complex models

3. **Gradient descent** efficiently finds optimal operating conditions

4. **`vmap` enables efficient sensitivity analysis** across parameter ranges

5. **Gradients scale well**: Cost is ~2x function evaluation regardless of dimension

6. **Advanced features**:
   - Jacobians for multi-output systems
   - Hessians for Newton-type optimization
   - JIT compilation for speed

## Next Steps

- Integrate with scipy.optimize for constrained optimization
- Use JAX's optax library for advanced optimizers (Adam, L-BFGS)
- Combine with neural networks for hybrid models